# Supp Fig 4d: composition of cell-type neighborhoods CT_0 and CT_3

**Reconstruction.** No code for this panel was found (repo history on all branches, EFS
Final_Notebooks, Analysis_Repo incl. Branch_*, results/, figures/ incl. zips, other clones);
the supplementary figure image is not in the manuscript PDF. This notebook implements the
Methods ("Neighborhood cell type composition and marker scoring"):

- composition of a neighborhood = cell2location abundances summed per cell type over its spots,
  row-normalised; doublet / other / unknown types excluded;
- Supp Fig 4d: log2 fold change of each cell type's composition in CT_0 and in CT_3 relative to
  its mean composition across the other eight neighborhoods; cell types shown = union of the
  `TOP_N` most enriched and most depleted of each of the two neighborhoods, restricted to those
  reaching ≥ 0.3 % composition;
- compartment numbers: aggregate composition of lymphoid + dendritic and of mesenchymal /
  malignant types in CT_0 minus all other neighborhoods, in percentage points.

Comparison with the text: "+7.1 percentage points" (lymphoid and dendritic, CT_0 vs other
neighborhoods) is reproduced exactly (+7.09) with the q05 abundances (the
`deconv_cohort_level_4` slot that `load_spatial_adata` reads; identical to the
`q05_cell_abundance_w_sf` of the cell2location files), level-1 Lymphoid types plus the DC
types, and "all other neighborhoods" pooled (abundance of all their spots). The "−6.7
percentage points" depletion of the mesenchymal program is not reproduced by any natural
compartment definition tried (printed below: MES-like −8.3, MES-like + Malignant_MES2-like
−11.6, all malignant −16.2 under the same settings); the definition used is unknown.
Neighborhood labels: the frozen k = 10 cell-type neighborhoods (`config.CT_ANNOTATIONS_CSV`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import config, output

In [ ]:
OUT = output.setup("suppfig4d_CT0_CT3_composition")
ABUNDANCE = "q05_cell_abundance_w_sf"   # or "means_cell_abundance_w_sf"
EXCLUDE_PATTERNS = ["doublet", "other", "unknown"]
FOCUS = ["0", "3"]
TOP_N = 5                 # most enriched and most depleted cell types per focus neighborhood
MIN_COMPOSITION = 0.003   # 0.3 %
EPS = 1e-6
PUBLISHED_PP = {"lymphoid + dendritic": 7.1, "mesenchymal": -6.7}

## Abundances per neighborhood

In [ ]:
ct_annot = pd.read_csv(config.CT_ANNOTATIONS_CSV, index_col="spot_id")
sections = sorted(ct_annot["sample"].unique())
c2l = AllDataCenters.load_cell2location(sections, abundance=ABUNDANCE).drop(columns="total_cells")
c2l.index = [f"{s}_{b}" for s, b in c2l.index]
c2l.columns = [c.split("_w_sf_", 1)[1] for c in c2l.columns]
c2l = c2l[[c for c in c2l.columns if not any(p in c.lower() for p in EXCLUDE_PATTERNS)]]

nh = ct_annot["celltype_neighborhood"].reindex(c2l.index).astype("Int64").astype("string")
valid = nh.notna() & (c2l.sum(axis=1) > 0)
abund = c2l[valid].groupby(nh[valid]).sum()
composition = abund.div(abund.sum(axis=1), axis=0)
print(f"Sections: {len(sections)}, spots: {int(valid.sum())}, cell types: {abund.shape[1]}, "
      f"neighborhoods: {list(abund.index)}")

## Compartments: CT_0 vs all other neighborhoods (percentage points)

In [ ]:
key = pd.read_csv(config.ANNOTATION_KEY_CSV).drop_duplicates("celltype_level4_scanvi").set_index("celltype_level4_scanvi")
level1 = key["celltype_level1_curated"].reindex(composition.columns)
cts = composition.columns
compartments = {
    "lymphoid + dendritic": ((level1 == "Lymphoid") | cts.str.contains("DC")).to_numpy(),
    "MES-like": cts.isin(["MES-like"]),
    "MES-like + Malignant_MES2-like": cts.isin(["MES-like", "Malignant_MES2-like"]),
    "malignant": (level1 == "Malignant").to_numpy(),
}
others = [n for n in abund.index if n != "0"]
pooled_others = abund.loc[others].sum() / abund.loc[others].values.sum()
pp = pd.DataFrame({
    name: {"CT_0": 100 * composition.loc["0", m].sum(), "other neighborhoods": 100 * pooled_others[m].sum(),
           "difference (pp)": 100 * (composition.loc["0", m].sum() - pooled_others[m].sum()),
           "cell types": ", ".join(cts[m])}
    for name, m in compartments.items()}).T
print(pp.drop(columns="cell types").astype(float).round(2).to_string())
print(f"\nPublished: lymphoid + dendritic {PUBLISHED_PP['lymphoid + dendritic']:+.1f} pp, "
      f"mesenchymal {PUBLISHED_PP['mesenchymal']:+.1f} pp")
output.save_table(OUT, "ct0_compartments_pp", pp)

## Supp Fig 4d: log2 fold change of CT_0 and CT_3 vs the other eight neighborhoods

In [ ]:
rest = [n for n in composition.index if n not in FOCUS]
ref = composition.loc[rest].mean()
log2fc = np.log2((composition.loc[FOCUS] + EPS).div(ref + EPS, axis=1))

shown = []
for n in FOCUS:
    ok = composition.loc[n] >= MIN_COMPOSITION
    lf = log2fc.loc[n, ok].sort_values()
    shown += lf.tail(TOP_N).index.tolist() + lf.head(TOP_N).index.tolist()
shown = list(dict.fromkeys(shown))
table = pd.DataFrame({f"log2FC CT_{n}": log2fc.loc[n, shown] for n in FOCUS} |
                     {f"composition CT_{n}": composition.loc[n, shown] for n in FOCUS} |
                     {"mean composition, other 8": ref[shown]})
table = table.sort_values(f"log2FC CT_{FOCUS[0]}", ascending=False)
print(table.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(9, 0.32 * len(table) + 1.5), sharey=True)
for ax, n in zip(axes, FOCUS):
    v = table[f"log2FC CT_{n}"]
    ax.barh(range(len(v)), v, color=np.where(v > 0, "#C0392B", "#2C7FB8"))
    ax.axvline(0, color="#7F8C8D", lw=0.8)
    ax.set_title(f"CT_{n}", fontsize=11)
    ax.set_xlabel("log2 FC vs mean of other 8 neighborhoods")
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
axes[0].set_yticks(range(len(table)))
axes[0].set_yticklabels(table.index, fontsize=8)
axes[0].invert_yaxis()
plt.tight_layout()
output.save(OUT, "ct0_ct3_log2fc_composition", fig, table, panel="Supp Fig 4d")
plt.show()

In [ ]:
print(output.versions(["numpy", "pandas", "matplotlib"]).to_string(index=False))